# Live Demo

This notebook provides separate live demos for:
- Phase 1 (`project_description_v1.0.pdf`): Phi-2 JSON agent
- Phase 2 (`project_description_v2.0.pdf`): Mistral ReAct agent
- Phase 2 Bonus: Mistral ReAct + DPO adapter

What each demo returns:
- the actions predicted by the model
- the model's own answer
- the final answer recomputed by running those actions with `ToolExecutor`

Before running the demos, update the config cell so `PROJECT_FILES_ROOT` points to the Drive folder that contains `sales_data.csv` and the helper files, while the phase roots point to the folders that contain the checkpoints.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DEPS_SENTINEL = Path('/tmp/.csf425_live_demo_deps_ready')
PACKAGES = [
    'transformers>=4.41.0',
    'peft>=0.10.0',
    'accelerate>=0.29.3',
    'bitsandbytes>=0.44.0',
    'packaging',
    'pandas',
]

if not DEPS_SENTINEL.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *PACKAGES])
    DEPS_SENTINEL.write_text('ok')
    print('Dependencies installed. Restarting the runtime once...')
    os.kill(os.getpid(), 9)

print('Dependencies already prepared for this runtime.')


In [ ]:
import gc
import glob
import json
import os
import re
from datetime import datetime

import pandas as pd
import torch
from packaging import version
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList

import transformers as _tf
if version.parse(_tf.__version__) < version.parse('4.41.0'):
    raise RuntimeError(f'transformers=={_tf.__version__} is too old. Re-run the dependency cell and let the runtime restart.')

assert torch.cuda.is_available(), 'No GPU detected. In Colab, switch to a T4 GPU runtime.'
print(f'transformers: {_tf.__version__}')
print(f'GPU         : {torch.cuda.get_device_name(0)}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

# Project files root: sales_data.csv, tool_executor.py, run_pipeline.py, agent_trajectories_2k.json.
PROJECT_FILES_ROOT = '/content/drive/MyDrive/CS_F425_Project'

# Checkpoint roots: edit these if the trained weights live in different Drive folders.
PHASE1_ROOT = '/content/drive/MyDrive/CS_F425_Project'
PHASE2_ROOT = '/content/drive/MyDrive/CS_F425_Project'
BONUS_ROOT = PHASE2_ROOT

# Optional: set this explicitly to a held-out CSV during evaluation.
CSV_PATH = None

PHASE1_BASE_MODEL = 'microsoft/phi-2'
PHASE2_BASE_MODEL = 'mistralai/Mistral-7B-v0.1'
PHASE1_MAX_SEQ_LENGTH = 384
PHASE2_MAX_SEQ_LENGTH = 768

PHASE1_ADAPTER_DIR = f'{PHASE1_ROOT}/phi2-agent-adapter'
PHASE1_CKPT_GLOB = f'{PHASE1_ROOT}/phi2-agent-qlora/checkpoint-*'
PHASE1_TIMED_GLOB = f'{PHASE1_ROOT}/phi2-agent-adapter/timed_ckpt_step_*'

PHASE2_ADAPTER_DIR = f'{PHASE2_ROOT}/mistral-react-adapter'
PHASE2_CKPT_GLOB = f'{PHASE2_ROOT}/mistral-react-qlora/checkpoint-*'
PHASE2_TIMED_GLOB = f'{PHASE2_ROOT}/mistral-react-adapter/timed_ckpt_step_*'

BONUS_ADAPTER_DIR = f'{BONUS_ROOT}/mistral-react-dpo-adapter'
BONUS_CKPT_GLOB = f'{BONUS_ROOT}/mistral-react-dpo-ckpt/checkpoint-*'

print('Project root :', PROJECT_FILES_ROOT)
print('Phase 1 root :', PHASE1_ROOT)
print('Phase 2 root :', PHASE2_ROOT)
print('Bonus root   :', BONUS_ROOT)

def resolve_csv_path(explicit_path=None):
    candidates = []
    if explicit_path:
        candidates.append(explicit_path)
    candidates.extend([
        f'{PROJECT_FILES_ROOT}/sales_data.csv',
        f'{PHASE1_ROOT}/sales_data.csv',
        f'{PHASE2_ROOT}/sales_data.csv',
        f'{BONUS_ROOT}/sales_data.csv',
        '/content/sales_data.csv',
        'sales_data.csv',
    ])
    seen = set()
    for path in candidates:
        if not path or path in seen:
            continue
        seen.add(path)
        if os.path.isfile(path):
            return path
    raise FileNotFoundError('Could not find sales_data.csv. Set CSV_PATH in this cell before continuing.')

CSV_PATH = resolve_csv_path(CSV_PATH)
df = pd.read_csv(CSV_PATH)
print('CSV path     :', CSV_PATH)
print('CSV shape    :', df.shape)
print('Helper files :', {
    'tool_executor.py': os.path.isfile(f'{PROJECT_FILES_ROOT}/tool_executor.py'),
    'run_pipeline.py': os.path.isfile(f'{PROJECT_FILES_ROOT}/run_pipeline.py'),
    'agent_trajectories_2k.json': os.path.isfile(f'{PROJECT_FILES_ROOT}/agent_trajectories_2k.json'),
})

class ToolExecutor:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def execute(self, actions):
        df_local = self.df
        for step_id, action in enumerate(actions):
            tool = action['tool']
            args = action.get('args', {})
            try:
                if tool == 'filter':
                    df_local = self._filter(df_local, **args)
                elif tool == 'groupby':
                    df_local = self._groupby(df_local, **args)
                elif tool == 'aggregate':
                    df_local = self._aggregate(df_local, **args)
                elif tool == 'sort':
                    df_local = self._sort(df_local, **args)
                elif tool == 'topk':
                    df_local = self._topk(df_local, **args)
                else:
                    raise ValueError(f'Unknown tool: {tool}')
            except Exception as exc:
                raise RuntimeError(f'Error at step {step_id} ({tool}): {exc}')
        return df_local

    def _filter(self, df_local, column, op, value):
        if op == '==':
            return df_local[df_local[column] == value]
        if op == '>':
            return df_local[df_local[column] > value]
        if op == '<':
            return df_local[df_local[column] < value]
        raise ValueError(f'Unsupported op: {op}')

    def _groupby(self, df_local, column):
        return df_local.groupby(column)

    def _aggregate(self, df_local, column, agg):
        if agg == 'sum':
            result = df_local[column].sum()
        elif agg == 'mean':
            result = df_local[column].mean()
        elif agg == 'count':
            result = df_local[column].count()
        else:
            raise ValueError(f'Unsupported aggregation: {agg}')
        if hasattr(result, 'reset_index'):
            return result.reset_index()
        return pd.DataFrame({column: [result]})

    def _sort(self, df_local, column, ascending=False):
        return df_local.sort_values(by=column, ascending=ascending)

    def _topk(self, df_local, k):
        return df_local.head(k)

def _parse_numeric(val_str):
    return float(val_str) if '.' in val_str else int(val_str)

def parse_agent_action(action_str):
    if action_str.startswith('filter_data'):
        match = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.[\d]+)?)", action_str)
        if match:
            return {
                'tool': 'filter',
                'args': {'column': match.group(1), 'op': '==', 'value': _parse_numeric(match.group(2))},
            }
        match = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if match:
            return {
                'tool': 'filter',
                'args': {'column': match.group(1), 'op': '==', 'value': match.group(2)},
            }
    elif action_str.startswith('group_by'):
        match = re.search(r"column='([^']+)'", action_str)
        if match:
            return {'tool': 'groupby', 'args': {'column': match.group(1)}}
    elif action_str.startswith('aggregate_sum'):
        match = re.search(r"column='([^']+)'", action_str)
        if match:
            return {'tool': 'aggregate', 'args': {'column': match.group(1), 'agg': 'sum'}}
    elif action_str.startswith('aggregate_mean'):
        match = re.search(r"column='([^']+)'", action_str)
        if match:
            return {'tool': 'aggregate', 'args': {'column': match.group(1), 'agg': 'mean'}}
    elif action_str.startswith('aggregate_count'):
        match = re.search(r"column='([^']+)'", action_str)
        if match:
            return {'tool': 'aggregate', 'args': {'column': match.group(1), 'agg': 'count'}}
    elif action_str.startswith('sort_by'):
        match = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if match:
            return {
                'tool': 'sort',
                'args': {'column': match.group(1), 'ascending': match.group(2) != 'desc'},
            }
    elif action_str.startswith('top_k'):
        match = re.search(r'k=(\d+)', action_str)
        if match:
            return {'tool': 'topk', 'args': {'k': int(match.group(1))}}
    return None

def clean_scalar(value):
    if hasattr(value, 'item'):
        try:
            value = value.item()
        except Exception:
            pass
    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, int):
        return int(value)
    if isinstance(value, float):
        return round(float(value), 4)
    return value

def result_to_python(actions, result_df):
    if result_df is None or (hasattr(result_df, 'empty') and result_df.empty):
        return None
    action_names = [action.split('(')[0] for action in actions]
    if getattr(result_df, 'shape', None) == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])
    has_groupby = any(name.startswith('group_by') for name in action_names)
    has_sort_or_topk = any(name.startswith('sort_by') or name.startswith('top_k') for name in action_names)
    if getattr(result_df, 'shape', (0, 0))[1] == 2 and has_groupby and not has_sort_or_topk:
        key_col, value_col = result_df.columns
        return {str(row[key_col]): clean_scalar(row[value_col]) for _, row in result_df.iterrows()}
    return [
        {str(key): clean_scalar(value) for key, value in row.items()}
        for row in result_df.to_dict(orient='records')
    ]

def compute_answer(actions, source_df):
    parsed = []
    for action in actions:
        try:
            parsed_action = parse_agent_action(action)
        except Exception:
            parsed_action = None
        if parsed_action is not None:
            parsed.append(parsed_action)
    if not parsed:
        return None
    try:
        result = ToolExecutor(source_df.copy()).execute(parsed)
        return result_to_python(actions, result)
    except Exception:
        return None

def pretty_print(payload):
    print(json.dumps(payload, indent=2, ensure_ascii=False))

def find_latest_peft_artifact(label, final_dir, patterns):
    candidates = []
    if final_dir and os.path.isfile(os.path.join(final_dir, 'adapter_config.json')):
        candidates.append({'kind': 'final_adapter', 'path': final_dir})
    for pattern in patterns:
        for path in glob.glob(pattern):
            if os.path.isfile(os.path.join(path, 'adapter_config.json')):
                candidates.append({'kind': os.path.basename(os.path.dirname(path)), 'path': path})
    if not candidates:
        raise FileNotFoundError(f'[{label}] No adapter or checkpoint with adapter_config.json was found.')
    dedup = {item['path']: item for item in candidates}
    ranked = sorted(dedup.values(), key=lambda item: os.path.getmtime(item['path']), reverse=True)
    print(f'[{label}] latest artifacts:')
    for item in ranked[:5]:
        stamp = datetime.fromtimestamp(os.path.getmtime(item['path'])).strftime('%Y-%m-%d %H:%M:%S')
        print(f"  - {item['path']}  ({item['kind']}, {stamp})")
    return ranked[0]

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
)
print('Compute dtype:', compute_dtype)

ACTIVE_BUNDLE = None

def clear_active_model():
    global ACTIVE_BUNDLE
    if ACTIVE_BUNDLE is not None:
        model = ACTIVE_BUNDLE.get('model')
        tokenizer = ACTIVE_BUNDLE.get('tokenizer')
        del model
        del tokenizer
        ACTIVE_BUNDLE = None
        gc.collect()
        torch.cuda.empty_cache()
        print('Cleared the previous model from GPU memory.')

def load_peft_bundle(key, base_model_name, adapter_path):
    global ACTIVE_BUNDLE
    if ACTIVE_BUNDLE and ACTIVE_BUNDLE['key'] == key and ACTIVE_BUNDLE['adapter_path'] == adapter_path:
        print(f'Reusing already loaded {key} model.')
        return ACTIVE_BUNDLE
    clear_active_model()
    tokenizer_source = adapter_path if os.path.isfile(os.path.join(adapter_path, 'tokenizer_config.json')) else base_model_name
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    print(f'Loading base model: {base_model_name}')
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=BNB_CONFIG,
        device_map='auto',
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    model.config.use_cache = True
    ACTIVE_BUNDLE = {
        'key': key,
        'base_model_name': base_model_name,
        'adapter_path': adapter_path,
        'model': model,
        'tokenizer': tokenizer,
    }
    print(f'Loaded {key} adapter from: {adapter_path}')
    return ACTIVE_BUNDLE

def extract_json_payload(raw_text):
    text = raw_text.strip()
    for candidate in [text, re.sub(r'```json\s*', '', text), re.sub(r'```\s*', '', text)]:
        candidate = candidate.strip()
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        candidate = re.sub(r',\s*([}\]])', r'\1', match.group()).strip()
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    return {'raw_output': raw_text}


In [ ]:
PHASE1_SCHEMA = '''Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  aggregate_mean(column='col')
  aggregate_count(column='col')
  sort_by(column='col', order='asc'|'desc')
  top_k(k=N)'''

PHASE1_PROMPT_TEMPLATE = '''### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an "actions" list and an "answer" field. No other text.

### Schema
{schema}

### Question
{question}

### Answer
'''

def make_phase1_prompt(question):
    return PHASE1_PROMPT_TEMPLATE.format(schema=PHASE1_SCHEMA, question=question)

def load_phase1_bundle():
    artifact = find_latest_peft_artifact(
        'Phase 1',
        PHASE1_ADAPTER_DIR,
        [PHASE1_CKPT_GLOB, PHASE1_TIMED_GLOB],
    )
    return load_peft_bundle('phase1', PHASE1_BASE_MODEL, artifact['path'])

def generate_phase1(question, max_new_tokens=200):
    if not question:
        raise ValueError('Query cannot be empty.')
    bundle = load_phase1_bundle()
    prompt = make_phase1_prompt(question)
    inputs = bundle['tokenizer'](prompt, return_tensors='pt').to(bundle['model'].device)
    with torch.no_grad():
        output_ids = bundle['model'].generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=bundle['tokenizer'].eos_token_id,
            eos_token_id=bundle['tokenizer'].eos_token_id,
        )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    raw_output = bundle['tokenizer'].decode(new_tokens, skip_special_tokens=True).strip()
    return extract_json_payload(raw_output), raw_output, bundle['adapter_path']

def phase1_live_demo(question=None):
    if question is None:
        question = input('Phase 1 query: ').strip()
    result, raw_output, adapter_path = generate_phase1(question)
    actions = result.get('actions', [])
    executor_answer = compute_answer(actions, df) if actions else None
    payload = {
        'query': question,
        'weights_path': adapter_path,
        'actions': actions,
        'model_answer': result.get('answer'),
        'answer': executor_answer,
    }
    if 'raw_output' in result:
        payload['raw_output'] = raw_output
    pretty_print(payload)
    return payload


In [ ]:
phase1_live_demo()


In [ ]:
REACT_SYSTEM_PROMPT = '''You are a data analysis agent. You have access to the following tools to analyze a sales dataset:

Function Descriptions:
[
  {"name": "filter_data", "description": "Filter rows where column equals value", "parameters": {"column": "str", "value": "str or int"}},
  {"name": "group_by", "description": "Group the data by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_mean", "description": "Average a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_count", "description": "Count rows for a column", "parameters": {"column": "str"}},
  {"name": "sort_by", "description": "Sort by a column", "parameters": {"column": "str", "order": "asc or desc"}},
  {"name": "top_k", "description": "Select top k rows", "parameters": {"k": "int"}}
]

Schema: date (date), year (int), month (int), city (str), region (str), product (str), category (str), revenue (float), units_sold (int), cost (float), profit (float)

Use the Thought/Action/Action Input format. Wait for Observation after each action. End with Final Answer.'''

REACT_PROMPT_TEMPLATE = '''### System
{system}

### User Query
{question}

### Agent Scratchpad
{scratchpad}'''

def format_observation(result_df, max_chars=500):
    if result_df is None:
        return 'No data returned.'
    if hasattr(result_df, 'empty') and result_df.empty:
        return 'Empty result.'
    if getattr(result_df, 'shape', None) == (1, 1):
        return str(result_df.iloc[0, 0])
    text = result_df.to_string(index=False)
    if len(text) > max_chars:
        text = text[:max_chars] + f'\n... (truncated, {result_df.shape[0]} rows total)'
    return text

class ObservationStopCriteria(StoppingCriteria):
    def __init__(self, tokenizer, triggers=('Observation:', 'Final Answer:')):
        self.tokenizer = tokenizer
        self.triggers = triggers

    def __call__(self, input_ids, scores, **kwargs):
        tail = self.tokenizer.decode(input_ids[0, -30:], skip_special_tokens=True)
        return any(trigger in tail for trigger in self.triggers)

def sanitize_json(text):
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    text = re.sub(r',\s*([}\]])', r'\1', text)
    return text.strip()

class ReActAgent:
    def __init__(self, model, tokenizer, source_df, max_steps=5, max_obs_chars=500, max_consecutive_errors=2, max_new_tokens=256):
        self.model = model
        self.tokenizer = tokenizer
        self.source_df = source_df
        self.max_steps = max_steps
        self.max_obs_chars = max_obs_chars
        self.max_consecutive_errors = max_consecutive_errors
        self.max_new_tokens = max_new_tokens
        self.stop_criteria = StoppingCriteriaList([ObservationStopCriteria(tokenizer)])

    def run(self, question):
        self.model.eval()
        self.model.config.use_cache = True
        current_df = self.source_df.copy()
        scratchpad = ''
        consecutive_errors = 0
        all_actions = []
        for step in range(self.max_steps):
            prompt = REACT_PROMPT_TEMPLATE.format(
                system=REACT_SYSTEM_PROMPT,
                question=question,
                scratchpad=scratchpad,
            )
            output_text = self._generate(prompt)
            scratchpad += output_text
            if 'Final Answer:' in output_text:
                answer_text = output_text.split('Final Answer:')[-1].strip()
                answer_text = answer_text.replace(self.tokenizer.eos_token or '', '').strip()
                return {
                    'actions': all_actions,
                    'answer': self._parse_final_answer(answer_text),
                    'steps': step + 1,
                    'scratchpad': scratchpad,
                }
            action_name, action_input = self._parse_action(output_text)
            if action_name is None:
                consecutive_errors += 1
                error_msg = 'Could not parse action from output. Use format: Action: <name>\\nAction Input: <args>'
                scratchpad += f'\nObservation: ERROR - {error_msg}\n'
                if consecutive_errors >= self.max_consecutive_errors:
                    return {
                        'actions': all_actions,
                        'answer': None,
                        'error': f'Max consecutive errors ({self.max_consecutive_errors}) reached.',
                        'steps': step + 1,
                        'scratchpad': scratchpad,
                    }
                continue
            consecutive_errors = 0
            action_str = f'{action_name}({action_input})'
            all_actions.append(action_str)
            try:
                parsed = parse_agent_action(action_str)
                if parsed is None:
                    raise ValueError(f'Unknown action: {action_str}')
                result = ToolExecutor(current_df).execute([parsed])
                obs = format_observation(result, max_chars=self.max_obs_chars)
                if isinstance(result, pd.DataFrame) and not result.empty:
                    current_df = result
            except Exception as exc:
                obs = f'ERROR - {type(exc).__name__}: {exc}'
            scratchpad += f'\nObservation: {obs}\n'
        return {
            'actions': all_actions,
            'answer': None,
            'error': f'Max steps ({self.max_steps}) reached without Final Answer.',
            'steps': self.max_steps,
            'scratchpad': scratchpad,
        }

    def _generate(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=PHASE2_MAX_SEQ_LENGTH).to(self.model.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=self.stop_criteria,
            )
        new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def _parse_action(self, text):
        action_match = re.search(r'Action:\s*(\w+)', text)
        input_match = re.search(r'Action Input:\s*(.+?)(?:\n|$)', text, re.DOTALL)
        if action_match and input_match:
            return action_match.group(1).strip(), input_match.group(1).strip()
        return None, None

    def _parse_final_answer(self, text):
        text = sanitize_json(text)
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
        for pattern in [r'\{[^{}]*\}', r'\[.*\]']:
            match = re.search(pattern, text, re.DOTALL)
            if match:
                try:
                    return json.loads(sanitize_json(match.group()))
                except json.JSONDecodeError:
                    pass
        num_match = re.search(r'-?[\d]+(?:\.[\d]+)?', text)
        if num_match:
            val = num_match.group()
            return float(val) if '.' in val else int(val)
        return text

def load_phase2_bundle():
    artifact = find_latest_peft_artifact(
        'Phase 2',
        PHASE2_ADAPTER_DIR,
        [PHASE2_CKPT_GLOB, PHASE2_TIMED_GLOB],
    )
    bundle = load_peft_bundle('phase2', PHASE2_BASE_MODEL, artifact['path'])
    bundle['agent'] = ReActAgent(bundle['model'], bundle['tokenizer'], df, max_steps=5)
    return bundle

def load_bonus_bundle():
    artifact = find_latest_peft_artifact(
        'Phase 2 Bonus',
        BONUS_ADAPTER_DIR,
        [BONUS_CKPT_GLOB],
    )
    bundle = load_peft_bundle('bonus', PHASE2_BASE_MODEL, artifact['path'])
    bundle['agent'] = ReActAgent(bundle['model'], bundle['tokenizer'], df, max_steps=5)
    return bundle

def react_live_demo(loader, label, question=None, show_scratchpad=False):
    if question is None:
        question = input(f'{label} query: ').strip()
    if not question:
        raise ValueError('Query cannot be empty.')
    bundle = loader()
    result = bundle['agent'].run(question)
    actions = result.get('actions', [])
    executor_answer = compute_answer(actions, df) if actions else None
    payload = {
        'query': question,
        'weights_path': bundle['adapter_path'],
        'actions': actions,
        'model_answer': result.get('answer'),
        'answer': executor_answer,
        'steps': result.get('steps'),
    }
    if result.get('error'):
        payload['error'] = result['error']
    pretty_print(payload)
    if show_scratchpad:
        print('\nScratchpad:\n')
        print(result.get('scratchpad', ''))
    return payload


In [ ]:
react_live_demo(load_phase2_bundle, 'Phase 2')


In [ ]:
react_live_demo(load_bonus_bundle, 'Phase 2 Bonus')
